# **Fine Tune With Adapters**

#### **Import Libraries**

In [2]:
import torch
import torch.nn as nn 
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, BertForSequenceClassification
from tqdm import tqdm
from torch.utils.data import Dataset
import math

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### **Dataset Configuration**

In [3]:
dataset = load_dataset("imdb")
dataset

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 331784.28 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [9]:
train_dataset = dataset['train']
test_dataset = dataset['test']

In [21]:
train_dataset[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [24]:
label_list = []
for id , (text, label) in enumerate(train_dataset):
    label_list.append(train_dataset[id]['label'])

num_labels = len(set(label_list))
num_labels    

2

In [25]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [27]:
lmdb_labels = {0: "negatice review", 1: "positive review"}

In [35]:
label_col = train_dataset.select_columns('label')
label_col

Dataset({
    features: ['label'],
    num_rows: 25000
})

### **Define Dataset Class**

In [ ]:
class ClassificationDataset(Dataset):
    
    def __init__(self,dataset,tokenizer: AutoTokenizer):
        super().__init__()
        
        self.text = dataset.select_columns('text')
        self.label = dataset.select_columns('label')
        self.tokenizer = tokenizer
        self.dataset = dataset
    
        